## Ejercicio 11

Trabajo Práctico — Reconstrucción y regularización en imágenes tomográficas

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import scipy

In [2]:
X_fantoma = np.array([
    [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
    [0.0, 0.0, 0.8, 0.3, 0.3, 0.3, 0.0, 0.0],
    [0.0, 0.8, 0.8, 0.8, 0.3, 0.3, 0.3, 0.0],
    [0.0, 0.3, 0.8, 0.3, 0.3, 0.3, 0.3, 0.0],
    [0.0, 0.3, 0.3, 0.3, 0.3, 0.5, 0.3, 0.0],
    [0.0, 0.3, 0.3, 0.3, 0.5, 0.5, 0.5, 0.0],
    [0.0, 0.0, 0.3, 0.3, 0.3, 0.5, 0.0, 0.0],
    [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
], dtype=float)

n = X_fantoma.shape[0]
x_fantoma = X_fantoma.reshape(-1)

In [3]:
def mascara_rayo(n, direccion, s):
    alfa = direccion[0]
    beta = direccion[1]
    matriz = np.zeros((n, n), dtype=int)
    for i in range(n):
        for j in range(n):
            if alfa * i + beta * j == s:
                matriz[i][j] = 1
    return matriz


def matriz_proyeccion(n, direcciones):
    filas_A = []
    rayos = []

    for direccion in direcciones:
        alfa = direccion[0]
        beta = direccion[1]
        valores_s = set()

        for i in range(n):
            for j in range(n):
                valores_s.add(alfa * i + beta * j)

        for s in sorted(valores_s):
            mascara = mascara_rayo(n, direccion, s)
            filas_A.append(mascara.reshape(-1))
            rayos.append((direccion, s))

    A = np.array(filas_A, dtype=float)
    return A, rayos

In [4]:
D2 = [(1, 0), (0, 1)]
D4 = D2 + [(1, 1), (1, -1)]
D7 = D4 + [(2, 1), (1, 2), (3, 1)]

A2, rayos2 = matriz_proyeccion(n, D2)
A4, rayos4 = matriz_proyeccion(n, D4)
A7, rayos7 = matriz_proyeccion(n, D7)

In [5]:
def matriz_diferencias(n):
    filas_h = []
    filas_v = []

    for i in range(n):
        for j in range(n - 1):
            k = i * n + j
            fila = np.zeros(n**2)
            fila[k] = -1
            fila[k + 1] = 1
            filas_h.append(fila)

    for i in range(n - 1):
        for j in range(n):
            k = i * n + j
            fila = np.zeros(n**2)
            fila[k] = -1
            fila[k + n] = 1
            filas_v.append(fila)

    return np.array(filas_h + filas_v, dtype=float)

In [6]:
b7 = A7 @ x_fantoma
rng = np.random.default_rng(2026)
z = rng.standard_normal(b7.size)
escala = np.sqrt(np.mean(b7**2))

In [7]:
D = matriz_diferencias(n)
sigma = 0.06
b_sigma = b7 + sigma * escala * z
x_rec = np.linalg.lstsq(A7, b_sigma, rcond=None)[0]
r_sigma = b_sigma - A7 @ x_rec

### Inciso a)

La reconstrucción tomográfica es un problema inverso porque conocemos el efecto de la imagen
sobre los rayos y buscamos la imagen que lo produjo. Se conocen las mediciones $b$ y la matriz
de adquisición $A$; se busca el vector de atenuaciones $x$.

El modelo directo $b=Ax$ permite simular mediciones a partir de una imagen. El problema inverso
recorre esa relación en sentido contrario. En el fantoma hay 64 píxeles; en la aplicación final
hay 16.384 incógnitas y 16.470 mediciones. La cercanía entre esas cantidades no garantiza por
sí sola una reconstrucción estable.

### Inciso b)

In [8]:
print(f"{'Matriz':<8} {'Filas':>7} {'Columnas':>10} {'Rango':>8} {'Nulidad':>10}")
for nombre, A in [("A2", A2), ("A4", A4), ("A7", A7)]:
    rango = np.linalg.matrix_rank(A)
    print(f"{nombre:<8} {A.shape[0]:7d} {A.shape[1]:10d} {rango:8d} {A.shape[1]-rango:10d}")

Matriz     Filas   Columnas    Rango    Nulidad
A2            16         64       15         49
A4            46         64       39         25
A7           119         64       64          0


Para una imagen de $n^2$ píxeles, la unicidad en el sistema compatible requiere
$\operatorname{rango}(A)=n^2$, equivalente a $\operatorname{Nul}(A)=\{0\}$.
Si $Av=0$ con $v\ne0$, entonces $A(x+\gamma v)=Ax$: la adquisición no distingue esas imágenes.

En nuestros resultados, `A2` y `A4` tienen nulidades 49 y 25. Por eso admiten imágenes
distintas con las mismas mediciones. `A7` tiene rango 64 y nulidad cero, y recupera el fantoma
sin ruido. Una cantidad grande de filas puede incluir mediciones redundantes y no reemplaza
el análisis del rango.

### Inciso c)

In [9]:
print("σ:", sigma)
print("Norma del ruido:", np.linalg.norm(b_sigma - b7))
print("Norma del residuo:", np.linalg.norm(r_sigma))
print("Norma de A7.T @ r_sigma:", np.linalg.norm(A7.T @ r_sigma))
print("Error relativo:", np.linalg.norm(x_rec - x_fantoma) / np.linalg.norm(x_fantoma))

σ: 0.06
Norma del ruido: 0.733478799704043
Norma del residuo: 0.5320380148512261
Norma de A7.T @ r_sigma: 9.242410779200659e-14
Error relativo: 0.11333490950741712


El ruido modifica el modelo a $b_\sigma=Ax+\eta_\sigma$. En general esas mediciones quedan
fuera de $\operatorname{Col}(A)$ y el sistema deja de ser compatible. Mínimos cuadrados busca

$$\hat x=\arg\min_x\|Ax-b_\sigma\|^2.$$

El residuo $r=b_\sigma-A\hat x$ es la parte de las mediciones que el modelo no explica.
Se cumple $A^Tr\approx0$, de modo que $A\hat x$ es la proyección ortogonal de $b_\sigma$
sobre $\operatorname{Col}(A)$.

Con $\sigma=0{,}06$, el residuo es aproximadamente $0{,}532$ y el error relativo es $0{,}113$.
La norma del residuo sola no alcanza para juzgar la imagen: con `A2` el residuo sin ruido es
prácticamente cero, pero el error relativo es aproximadamente $0{,}469$. También puede reducirse
el residuo ajustando ruido, como se observa en la reconstrucción final sin regularización.

### Inciso d)

In [10]:
print(f"{'Sigma':>7} {'λ':>6} {'Error relativo':>17} {'Residuo':>12} {'Variación':>12}")
H = A7.T @ A7
for sigma_tabla in [0.0, 0.1, 0.2]:
    b_tabla = b7 + sigma_tabla * escala * z
    for lam in [0.0, 0.1, 0.3, 1.0]:
        x = np.linalg.solve(H + lam * D.T @ D, A7.T @ b_tabla)
        error = np.linalg.norm(x - x_fantoma) / np.linalg.norm(x_fantoma)
        print(f"{sigma_tabla:7.2f} {lam:6.1f} {error:17.6f} "
              f"{np.linalg.norm(A7 @ x - b_tabla):12.6f} {np.linalg.norm(D @ x):12.6f}")

  Sigma      λ    Error relativo      Residuo    Variación
   0.00    0.0          0.000000     0.000000     2.705550
   0.00    0.1          0.042271     0.200834     2.541775
   0.00    0.3          0.097224     0.504004     2.312200
   0.00    1.0          0.199992     1.159432     1.872875
   0.10    0.0          0.188892     0.886730     2.875522
   0.10    0.1          0.156938     0.916960     2.660041
   0.10    0.3          0.150892     1.049370     2.388844
   0.10    1.0          0.212752     1.522424     1.907656
   0.20    0.0          0.377783     1.773460     3.276282
   0.20    0.1          0.314809     1.797981     2.962689
   0.20    0.3          0.264377     1.900395     2.601386
   0.20    1.0          0.255698     2.276671     2.015785


El término $\|Dx\|^2$ introduce el supuesto de que los píxeles vecinos suelen tener
atenuaciones parecidas. Penaliza cambios locales, incluso los que pueden corresponder a bordes reales.

Sin ruido, la regularización empeora la recuperación exacta. Con ruido del 10 %, el mejor
valor probado en el fantoma es $\lambda=0{,}3$, que reduce el error de $0{,}189$ a $0{,}151$.
Con ruido del 20 %, el mejor de la grilla es $\lambda=1$, que baja el error de $0{,}378$ a $0{,}256$.
Así se ve el compromiso: mayor suavidad reduce parte del ruido, pero una penalización excesiva
borra detalles y aumenta el sesgo.

En la imagen desconocida elegimos $\lambda=30$ apoyándonos en el residuo, el ruido estimado y
las estructuras observadas. No comparamos directamente ese valor de $\lambda$ con los del
fantoma, porque cambiaron la escala y el modelo de la matriz de adquisición, además de la resolución.

### Inciso e)

In [11]:
def gradiente_descendiente(A, b, alpha, max_iter=10000, tol=1e-8,
                          D=None, lam=0.0, x_ref=None):
    x = np.zeros(A.shape[1], dtype=float)
    AT = A.T
    DT = None if D is None else D.T
    costos = []
    normas_gradiente = []
    errores = []
    norma_inicial = None
    motivo = "máximo de iteraciones"

    for t in range(max_iter + 1):
        residuo = A @ x - b
        costo = 0.5 * np.dot(residuo, residuo)
        grad = AT @ residuo

        if D is not None and lam != 0:
            diferencias = D @ x
            costo += 0.5 * lam * np.dot(diferencias, diferencias)
            grad = grad + lam * (DT @ diferencias)

        norma_grad = np.linalg.norm(grad)
        if norma_inicial is None:
            norma_inicial = norma_grad

        costos.append(float(costo))
        normas_gradiente.append(float(norma_grad))
        # La referencia se usa solamente para registrar el error, nunca para detenernos.
        if x_ref is not None:
            errores.append(np.linalg.norm(x - x_ref) / np.linalg.norm(x_ref))

        if not np.isfinite(costo) or costo > 1e25:
            motivo = "costo divergente"
            break
        if norma_grad <= tol * max(1.0, norma_inicial):
            motivo = "tolerancia del gradiente"
            break
        if t == max_iter:
            break

        x = x - alpha * grad

    return {"x": x, "costos": np.array(costos),
            "normas_gradiente": np.array(normas_gradiente),
            "errores": np.array(errores), "iteraciones": t, "motivo": motivo}

In [12]:
mu_max = np.linalg.eigvalsh(H)[-1]
resultado_gd = gradiente_descendiente(A7, b_sigma, 1.0 / mu_max, tol=1e-10)
x0 = np.zeros(n**2)
x_newton = x0 - np.linalg.solve(H, A7.T @ (A7 @ x0 - b_sigma))
x_lstsq = np.linalg.lstsq(A7, b_sigma, rcond=None)[0]

print("Iteraciones de gradiente:", resultado_gd["iteraciones"])
print("Pasos de Newton: 1")
print("Distancia gradiente - lstsq:", np.linalg.norm(resultado_gd["x"] - x_lstsq))
print("Distancia Newton - lstsq:", np.linalg.norm(x_newton - x_lstsq))
assert np.allclose(resultado_gd["x"], x_lstsq, atol=1e-7)
assert np.allclose(x_newton, x_lstsq, atol=1e-10)

Iteraciones de gradiente: 1149
Pasos de Newton: 1
Distancia gradiente - lstsq: 1.2993138953094514e-08
Distancia Newton - lstsq: 1.1982796382043795e-14


`lstsq` obtiene una solución de mínimos cuadrados sin seleccionar una tasa de aprendizaje.
Si no hay unicidad, devuelve la solución de norma mínima. Gradiente descendiente requiere
una tasa estable, una tolerancia y un máximo de iteraciones, pero trabaja mediante productos
matriz-vector, lo que permite aplicarlo a la matriz dispersa del ejercicio final.

Newton requiere resolver un sistema con la Hessiana. En esta cuadrática con `A7` de rango
columna completo, $H=A_7^TA_7$ es constante e invertible. El paso lleva directamente a
$H^{-1}A_7^Tb_\sigma$ desde cualquier punto inicial, por eso basta una iteración.
No se puede trasladar esa afirmación usando `solve` a una Hessiana singular sin adaptar el método.

Las tres soluciones coinciden dentro de la tolerancia. Ninguno de esos procedimientos elimina
por sí solo el ruido: para cambiar el compromiso entre ajuste y suavidad debemos cambiar el
problema, por ejemplo incorporando la regularización.

### Inciso f)

El modelo $b=Ax$ conecta la imagen con sus mediciones. El **rango** indica cuánta información
lineal independiente aporta la adquisición. El **espacio nulo** describe cambios de imagen
que los rayos no detectan y explica por qué puede faltar unicidad.

Cuando hay ruido, **mínimos cuadrados** busca la mejor aproximación compatible con el modelo.
La **proyección ortogonal** explica esa aproximación: separa las mediciones en un componente
de $\operatorname{Col}(A)$ y un residuo ortogonal. Las ecuaciones normales
$A^TA\hat x=A^Tb$ conectan esa geometría con el gradiente y la Hessiana del costo.

Los autovalores de $A^TA$ describen la curvatura y condicionan la tasa de gradiente descendiente.
Direcciones de poca curvatura pueden ser sensibles al ruido, aun cuando la solución sea única.
La **regularización** incorpora información adicional sobre la imagen mediante

$$\min_x\frac12\bigl(\|Ax-b\|^2+\lambda\|Dx\|^2\bigr),
\qquad (A^TA+\lambda D^TD)x=A^Tb.$$

Así se equilibra la fidelidad a las mediciones con la suavidad. El fantoma permite medir el
error real y estudiar ese equilibrio en condiciones controladas. Para una imagen desconocida,
la elección debe apoyarse en el nivel de ruido, el residuo y la conservación de las estructuras.

Asistencia utilizada: ChatGPT / Codex para completar y revisar el código, las explicaciones y las verificaciones de este notebook.